## SRP179635

**paper:** [PMID: 31289822](https://pmc.ncbi.nlm.nih.gov/articles/PMC6759067/) - Epimutations in Developmental Genes Underlie the Onset of Domestication in Farmed European Sea Bass, 2019

**date, curator:** 2026-06-08, Sara Carsanaro

**notes**
* rejected Reduced Representation Bisulfite Sequencing libraries

### annotation summary

In [22]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,Liver,UBERON:0002107,liver,perfect match
5,Brain,UBERON:0000955,brain,perfect match
20,Testis,UBERON:0000473,testis,perfect match
27,Skeletal muscle,UBERON:0001134,skeletal muscle tissue,perfect match


In [23]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,Adults,UBERON:0000113,post-juvenile adult stage,perfect match
10,3 years,UBERON:0000113,post-juvenile adult stage,missing child term


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP179635"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

## validation
path_to_v_script = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/validate_annotations.py'
path_to_rules = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/rules/'
val_output = "{}{}/validation.tsv".format(path_to_output_main, experiment_id)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
curl: (22) The requested URL returned error: 429
>78 ERROR: >78 curl command failed ( Mon Jun  8 14:55:43 CEST 2026 ) with: 22>78
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi -d db=pubmed&id=35741749&version=2.0&retmode=xml&tool=edirect&edirect=16.2&edirect_os=Darwin&email=scarsana%40SORGE42778>78
HTTP/1.0 429 Too Many Requests
nquire -url https://eutils.ncbi.nlm.nih.gov/entrez/eutils/ esummary.fcgi -db pubmed -id 35741749 -version 2.0 -retmode xml -tool edirect -edirect 16.2 -edirect_os Darwin -email scarsana@SORGE42778
EMPTY RESULT
SECOND ATTEMPT
0it [00:00, ?it/s]
curl:

### library annnotations

In [4]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX5252128,SRP179635,Illumina HiSeq 2000,SRS4253834,,,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563902,Liver,Adults,,,,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Liver5-RNAseq-wild,"SAMN10752196,GSM3563902",,,,,,,,wild,,,08/06/2026,"Trizol RNA extraction. mRNA-Seq sample preparation kit (TruSeq Stranded mRNA LT Sample Prep Kit, Illumina).",,,,Liver,,,,TRANSCRIPTOMIC,cDNA
1,SRX5252127,SRP179635,Illumina HiSeq 2000,SRS4253833,,,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563901,Liver,Adults,,,,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Liver4-RNAseq-wild,"SAMN10752197,GSM3563901",,,,,,,,wild,,,08/06/2026,"Trizol RNA extraction. mRNA-Seq sample preparation kit (TruSeq Stranded mRNA LT Sample Prep Kit, Illumina).",,,,Liver,,,,TRANSCRIPTOMIC,cDNA
2,SRX5252126,SRP179635,Illumina HiSeq 2000,SRS4253831,,,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563900,Liver,Adults,,,,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Liver3-RNAseq-wild,"SAMN10752198,GSM3563900",,,,,,,,wild,,,08/06/2026,"Trizol RNA extraction. mRNA-Seq sample preparation kit (TruSeq Stranded mRNA LT Sample Prep Kit, Illumina).",,,,Liver,,,,TRANSCRIPTOMIC,cDNA
3,SRX5252125,SRP179635,Illumina HiSeq 2000,SRS4253832,,,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563899,Liver,Adults,,,,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Liver2-RNAseq-wild,"SAMN10752199,GSM3563899",,,,,,,,wild,,,08/06/2026,"Trizol RNA extraction. mRNA-Seq sample preparation kit (TruSeq Stranded mRNA LT Sample Prep Kit, Illumina).",,,,Liver,,,,TRANSCRIPTOMIC,cDNA
4,SRX5252124,SRP179635,Illumina HiSeq 2000,SRS4253830,,,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563898,Liver,Adults,,,,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Liver1-RNAseq-wild,"SAMN10752200,GSM3563898",,,,,,,,wild,,,08/06/2026,"Trizol RNA extraction. mRNA-Seq sample preparation kit (TruSeq Stranded mRNA LT Sample Prep Kit, Illumina).",,,,Liver,,,,TRANSCRIPTOMIC,cDNA
5,SRX5252123,SRP179635,Illumina HiSeq 2000,SRS4253829,,,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563897,Brain,Adults,,,,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Brain5-RNAseq-wild,"SAMN10752201,GSM3563897",,,,,,,,wild,,,08/06/2026,"Trizol RNA extraction. mRNA-Seq sample preparation kit (TruSeq Stranded mRNA LT Sample Prep Kit, Illumina).",,,,Brain,,,,TRANSCRIPTOMIC,cDNA
6,SRX5252122,SRP179635,Illumina HiSeq 2000,SRS4253828,,,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563896,Brain,Adults,,,,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Brain4-RNAseq-wild,"SAMN10752202,GSM3563896",,,,,,,,wild,,,08/06/2026,"Trizol RNA extraction. mRNA-Seq sample preparation kit (TruSeq Stranded mRNA LT Sample Prep Kit, Illumina).",,,,Brain,,,,TRANSCRIPTOMIC,cDNA
7,SRX5252121,SRP179635,Illumina HiSeq 2000,SRS4253827,,,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563895,Brain,Adults,,,,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Brain3-RNAseq-wild,"SAMN10752203,GSM3563895",,,,,,,,wild,,,08/06/2026,"Trizol RNA extraction. mRNA-Seq sample preparation kit (TruSeq Stranded mRNA LT Sample Prep Kit, Illumina).",,,,Brain,,,,TRANSCRIPTOMIC,cDNA
8,SRX5252120,SRP179635,Illumina HiSeq 2000,SRS4253825,,,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563894,Brain,Adults,,,,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Brain2-RNAseq-wild,"SAMN10752204,GSM3563894",,,,,,,,wild,,,08

#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['Brain' 'Liver' 'Skeletal muscle' 'Testis']


In [6]:

# Brain
library.loc[library["infoOrgan"] == "Brain", "anatId"] = "UBERON:0000955"
library.loc[library["infoOrgan"] == "Brain", "anatName"] = "brain"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "Brain", "anatAnnotationStatus"] = "perfect match"
# full sampling, partial sampling, not documented
library.loc[library["infoOrgan"] == "Brain", "anatBiologicalStatus"] = "not documented"

# Liver
library.loc[library["infoOrgan"] == "Liver", "anatId"] = "UBERON:0002107"
library.loc[library["infoOrgan"] == "Liver", "anatName"] = "liver"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "Liver", "anatAnnotationStatus"] = "perfect match"
# full sampling, partial sampling, not documented
library.loc[library["infoOrgan"] == "Liver", "anatBiologicalStatus"] = "not documented"

# Skeletal muscle
library.loc[library["infoOrgan"] == "Skeletal muscle", "anatId"] = "UBERON:0001134"
library.loc[library["infoOrgan"] == "Skeletal muscle", "anatName"] = "skeletal muscle tissue"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "Skeletal muscle", "anatAnnotationStatus"] = "perfect match"
# full sampling, partial sampling, not documented
library.loc[library["infoOrgan"] == "Skeletal muscle", "anatBiologicalStatus"] = "not documented"

# Testis
library.loc[library["infoOrgan"] == "Testis", "anatId"] = "UBERON:0000473"
library.loc[library["infoOrgan"] == "Testis", "anatName"] = "testis"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "Testis", "anatAnnotationStatus"] = "perfect match"
# full sampling, partial sampling, not documented
library.loc[library["infoOrgan"] == "Testis", "anatBiologicalStatus"] = "not documented"

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX5252128,SRP179635,Illumina HiSeq 2000,SRS4253834,UBERON:0002107,liver,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563902,Liver,Adults,perfect match,not documented,,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Liver5-RNAseq-wild,"SAMN10752196,GSM3563902",,,,,,,,wild,,,08/06/2026,"Trizol RNA extraction. mRNA-Seq sample preparation kit (TruSeq Stranded mRNA LT Sample Prep Kit, Illumina).",,,,Liver,,,,TRANSCRIPTOMIC,cDNA
1,SRX5252127,SRP179635,Illumina HiSeq 2000,SRS4253833,UBERON:0002107,liver,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563901,Liver,Adults,perfect match,not documented,,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Liver4-RNAseq-wild,"SAMN10752197,GSM3563901",,,,,,,,wild,,,08/06/2026,"Trizol RNA extraction. mRNA-Seq sample preparation kit (TruSeq Stranded mRNA LT Sample Prep Kit, Illumina).",,,,Liver,,,,TRANSCRIPTOMIC,cDNA
2,SRX5252126,SRP179635,Illumina HiSeq 2000,SRS4253831,UBERON:0002107,liver,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563900,Liver,Adults,perfect match,not documented,,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Liver3-RNAseq-wild,"SAMN10752198,GSM3563900",,,,,,,,wild,,,08/06/2026,"Trizol RNA extraction. mRNA-Seq sample preparation kit (TruSeq Stranded mRNA LT Sample Prep Kit, Illumina).",,,,Liver,,,,TRANSCRIPTOMIC,cDNA
3,SRX5252125,SRP179635,Illumina HiSeq 2000,SRS4253832,UBERON:0002107,liver,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563899,Liver,Adults,perfect match,not documented,,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Liver2-RNAseq-wild,"SAMN10752199,GSM3563899",,,,,,,,wild,,,08/06/2026,"Trizol RNA extraction. mRNA-Seq sample preparation kit (TruSeq Stranded mRNA LT Sample Prep Kit, Illumina).",,,,Liver,,,,TRANSCRIPTOMIC,cDNA
4,SRX5252124,SRP179635,Illumina HiSeq 2000,SRS4253830,UBERON:0002107,liver,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563898,Liver,Adults,perfect match,not documented,,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Liver1-RNAseq-wild,"SAMN10752200,GSM3563898",,,,,,,,wild,,,08/06/2026,"Trizol RNA extraction. mRNA-Seq sample preparation kit (TruSeq Stranded mRNA LT Sample Prep Kit, Illumina).",,,,Liver,,,,TRANSCRIPTOMIC,cDNA
5,SRX5252123,SRP179635,Illumina HiSeq 2000,SRS4253829,UBERON:0000955,brain,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563897,Brain,Adults,perfect match,not documented,,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Brain5-RNAseq-wild,"SAMN10752201,GSM3563897",,,,,,,,wild,,,08/06/2026,"Trizol RNA extraction. mRNA-Seq sample preparation kit (TruSeq Stranded mRNA LT Sample Prep Kit, Illumina).",,,,Brain,,,,TRANSCRIPTOMIC,cDNA
6,SRX5252122,SRP179635,Illumina HiSeq 2000,SRS4253828,UBERON:0000955,brain,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563896,Brain,Adults,perfect match,not documented,,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Brain4-RNAseq-wild,"SAMN10752202,GSM3563896",,,,,,,,wild,,,08/06/2026,"Trizol RNA extraction. mRNA-Seq sample preparation kit (TruSeq Stranded mRNA LT Sample Prep Kit, Illumina).",,,,Brain,,,,TRANSCRIPTOMIC,cDNA
7,SRX5252121,SRP179635,Illumina HiSeq 2000,SRS4253827,UBERON:0000955,brain,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563895,Brain,Adults,perfect match,not documented,,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Brain3-RNAseq-wild,"SAMN10752203,GSM3563895",,,,,,,,wild,,,08/06/2026,"Trizol RNA extraction.

#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [7]:
unique_sorted(library, "infoStage")

['3 years' 'Adults']


In [8]:
# all
library.loc[:,'stageId'] = 'UBERON:0000113'
library.loc[:,'stageName'] = 'post-juvenile adult stage'



# perfect match, missing child term, other
library.loc[library["infoStage"] == "Adults", "stageAnnotationStatus"] = "perfect match"


# perfect match, missing child term, other
library.loc[library["infoStage"] == "3 years", "stageAnnotationStatus"] = "missing child term"


# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX5252128,SRP179635,Illumina HiSeq 2000,SRS4253834,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563902,Liver,Adults,perfect match,not documented,perfect match,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Liver5-RNAseq-wild,"SAMN10752196,GSM3563902",,,,,,,,wild,,,08/06/2026,"Trizol RNA extraction. mRNA-Seq sample preparation kit (TruSeq Stranded mRNA LT Sample Prep Kit, Illumina).",,,,Liver,,,,TRANSCRIPTOMIC,cDNA
1,SRX5252127,SRP179635,Illumina HiSeq 2000,SRS4253833,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563901,Liver,Adults,perfect match,not documented,perfect match,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Liver4-RNAseq-wild,"SAMN10752197,GSM3563901",,,,,,,,wild,,,08/06/2026,"Trizol RNA extraction. mRNA-Seq sample preparation kit (TruSeq Stranded mRNA LT Sample Prep Kit, Illumina).",,,,Liver,,,,TRANSCRIPTOMIC,cDNA
2,SRX5252126,SRP179635,Illumina HiSeq 2000,SRS4253831,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563900,Liver,Adults,perfect match,not documented,perfect match,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Liver3-RNAseq-wild,"SAMN10752198,GSM3563900",,,,,,,,wild,,,08/06/2026,"Trizol RNA extraction. mRNA-Seq sample preparation kit (TruSeq Stranded mRNA LT Sample Prep Kit, Illumina).",,,,Liver,,,,TRANSCRIPTOMIC,cDNA
3,SRX5252125,SRP179635,Illumina HiSeq 2000,SRS4253832,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563899,Liver,Adults,perfect match,not documented,perfect match,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Liver2-RNAseq-wild,"SAMN10752199,GSM3563899",,,,,,,,wild,,,08/06/2026,"Trizol RNA extraction. mRNA-Seq sample preparation kit (TruSeq Stranded mRNA LT Sample Prep Kit, Illumina).",,,,Liver,,,,TRANSCRIPTOMIC,cDNA
4,SRX5252124,SRP179635,Illumina HiSeq 2000,SRS4253830,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563898,Liver,Adults,perfect match,not documented,perfect match,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Liver1-RNAseq-wild,"SAMN10752200,GSM3563898",,,,,,,,wild,,,08/06/2026,"Trizol RNA extraction. mRNA-Seq sample preparation kit (TruSeq Stranded mRNA LT Sample Prep Kit, Illumina).",,,,Liver,,,,TRANSCRIPTOMIC,cDNA
5,SRX5252123,SRP179635,Illumina HiSeq 2000,SRS4253829,UBERON:0000955,brain,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563897,Brain,Adults,perfect match,not documented,perfect match,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Brain5-RNAseq-wild,"SAMN10752201,GSM3563897",,,,,,,,wild,,,08/06/2026,"Trizol RNA extraction. mRNA-Seq sample preparation kit (TruSeq Stranded mRNA LT Sample Prep Kit, Illumina).",,,,Brain,,,,TRANSCRIPTOMIC,cDNA
6,SRX5252122,SRP179635,Illumina HiSeq 2000,SRS4253828,UBERON:0000955,brain,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563896,Brain,Adults,perfect match,not documented,perfect match,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Brain4-RNAseq-wild,"SAMN10752202,GSM3563896",,,,,,,,wild,,,08/06/2026,"Trizol RNA extraction. mRNA-Seq sample preparation kit (TruSeq Stranded mRNA LT Sample Prep Kit, Illumin

#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [ ]:
#library.loc[:,'sex'] = ''
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [9]:
# making these variables because we use them again in the experiment file
my_protocol = 'TruSeq Stranded mRNA'
# full_length or 3'
my_protocolType = 'full_length'

library.loc[:,'protocol'] = my_protocol
library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX5252128,SRP179635,Illumina HiSeq 2000,SRS4253834,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563902,Liver,Adults,perfect match,not documented,perfect match,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Liver5-RNAseq-wild,"SAMN10752196,GSM3563902",,,,,,,,wild,,,08/06/2026,"Trizol RNA extraction. mRNA-Seq sample preparation kit (TruSeq Stranded mRNA LT Sample Prep Kit, Illumina).",,,,Liver,,,,TRANSCRIPTOMIC,cDNA
1,SRX5252127,SRP179635,Illumina HiSeq 2000,SRS4253833,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563901,Liver,Adults,perfect match,not documented,perfect match,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Liver4-RNAseq-wild,"SAMN10752197,GSM3563901",,,,,,,,wild,,,08/06/2026,"Trizol RNA extraction. mRNA-Seq sample preparation kit (TruSeq Stranded mRNA LT Sample Prep Kit, Illumina).",,,,Liver,,,,TRANSCRIPTOMIC,cDNA
2,SRX5252126,SRP179635,Illumina HiSeq 2000,SRS4253831,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563900,Liver,Adults,perfect match,not documented,perfect match,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Liver3-RNAseq-wild,"SAMN10752198,GSM3563900",,,,,,,,wild,,,08/06/2026,"Trizol RNA extraction. mRNA-Seq sample preparation kit (TruSeq Stranded mRNA LT Sample Prep Kit, Illumina).",,,,Liver,,,,TRANSCRIPTOMIC,cDNA
3,SRX5252125,SRP179635,Illumina HiSeq 2000,SRS4253832,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563899,Liver,Adults,perfect match,not documented,perfect match,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Liver2-RNAseq-wild,"SAMN10752199,GSM3563899",,,,,,,,wild,,,08/06/2026,"Trizol RNA extraction. mRNA-Seq sample preparation kit (TruSeq Stranded mRNA LT Sample Prep Kit, Illumina).",,,,Liver,,,,TRANSCRIPTOMIC,cDNA
4,SRX5252124,SRP179635,Illumina HiSeq 2000,SRS4253830,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563898,Liver,Adults,perfect match,not documented,perfect match,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Liver1-RNAseq-wild,"SAMN10752200,GSM3563898",,,,,,,,wild,,,08/06/2026,"Trizol RNA extraction. mRNA-Seq sample preparation kit (TruSeq Stranded mRNA LT Sample Prep Kit, Illumina).",,,,Liver,,,,TRANSCRIPTOMIC,cDNA
5,SRX5252123,SRP179635,Illumina HiSeq 2000,SRS4253829,UBERON:0000955,brain,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563897,Brain,Adults,perfect match,not documented,perfect match,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Brain5-RNAseq-wild,"SAMN10752201,GSM3563897",,,,,,,,wild,,,08/06/2026,"Trizol RNA extraction. mRNA-Seq sample preparation kit (TruSeq Stranded mRNA LT Sample Prep Kit, Illumina).",,,,Brain,,,,TRANSCRIPTOMIC,cDNA
6,SRX5252122,SRP179635,Illumina HiSeq 2000,SRS4253828,UBERON:0000955,brain,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563896,Brain,Adults,perfect match,not documented,perfect match,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Brain4-RNAseq-wild,"SAMN10752202,GSM3563896",,,,,,,,wild,,,08/06/2026,"Trizol RNA extraction. mRNA-Seq sample preparation kit (TruSeq Stranded mRNA LT Sample Prep Kit, Illumin

#### globin, replicates

In [10]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [ ]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [11]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-06-10'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX5252128,SRP179635,Illumina HiSeq 2000,SRS4253834,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563902,Liver,Adults,perfect match,not documented,perfect match,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Liver5-RNAseq-wild,"SAMN10752196,GSM3563902",,,,,,,,wild,,SAC,2026-06-10,"Trizol RNA extraction. mRNA-Seq sample preparation kit (TruSeq Stranded mRNA LT Sample Prep Kit, Illumina).",,,,Liver,,,,TRANSCRIPTOMIC,cDNA
1,SRX5252127,SRP179635,Illumina HiSeq 2000,SRS4253833,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563901,Liver,Adults,perfect match,not documented,perfect match,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Liver4-RNAseq-wild,"SAMN10752197,GSM3563901",,,,,,,,wild,,SAC,2026-06-10,"Trizol RNA extraction. mRNA-Seq sample preparation kit (TruSeq Stranded mRNA LT Sample Prep Kit, Illumina).",,,,Liver,,,,TRANSCRIPTOMIC,cDNA
2,SRX5252126,SRP179635,Illumina HiSeq 2000,SRS4253831,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563900,Liver,Adults,perfect match,not documented,perfect match,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Liver3-RNAseq-wild,"SAMN10752198,GSM3563900",,,,,,,,wild,,SAC,2026-06-10,"Trizol RNA extraction. mRNA-Seq sample preparation kit (TruSeq Stranded mRNA LT Sample Prep Kit, Illumina).",,,,Liver,,,,TRANSCRIPTOMIC,cDNA
3,SRX5252125,SRP179635,Illumina HiSeq 2000,SRS4253832,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563899,Liver,Adults,perfect match,not documented,perfect match,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Liver2-RNAseq-wild,"SAMN10752199,GSM3563899",,,,,,,,wild,,SAC,2026-06-10,"Trizol RNA extraction. mRNA-Seq sample preparation kit (TruSeq Stranded mRNA LT Sample Prep Kit, Illumina).",,,,Liver,,,,TRANSCRIPTOMIC,cDNA
4,SRX5252124,SRP179635,Illumina HiSeq 2000,SRS4253830,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563898,Liver,Adults,perfect match,not documented,perfect match,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Liver1-RNAseq-wild,"SAMN10752200,GSM3563898",,,,,,,,wild,,SAC,2026-06-10,"Trizol RNA extraction. mRNA-Seq sample preparation kit (TruSeq Stranded mRNA LT Sample Prep Kit, Illumina).",,,,Liver,,,,TRANSCRIPTOMIC,cDNA
5,SRX5252123,SRP179635,Illumina HiSeq 2000,SRS4253829,UBERON:0000955,brain,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563897,Brain,Adults,perfect match,not documented,perfect match,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Brain5-RNAseq-wild,"SAMN10752201,GSM3563897",,,,,,,,wild,,SAC,2026-06-10,"Trizol RNA extraction. mRNA-Seq sample preparation kit (TruSeq Stranded mRNA LT Sample Prep Kit, Illumina).",,,,Brain,,,,TRANSCRIPTOMIC,cDNA
6,SRX5252122,SRP179635,Illumina HiSeq 2000,SRS4253828,UBERON:0000955,brain,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563896,Brain,Adults,perfect match,not documented,perfect match,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Brain4-RNAseq-wild,"SAMN10752202,GSM3563896",,,,,,,,wild,,SAC,2026-06-10,"Trizol RNA extraction. mRNA-Seq sample preparation kit (TruSeq Stranded mRNA LT Sam

#### comments

In [12]:
library.loc[:,'comment'] = 'PMID: 31289822'

#### save complete file with correct columns

In [13]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX5252128,SRP179635,Illumina HiSeq 2000,SRS4253834,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563902,Liver,Adults,perfect match,not documented,perfect match,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Liver5-RNAseq-wild,"SAMN10752196,GSM3563902",,,,,,,PMID: 31289822,wild,,SAC,2026-06-10
1,SRX5252127,SRP179635,Illumina HiSeq 2000,SRS4253833,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563901,Liver,Adults,perfect match,not documented,perfect match,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Liver4-RNAseq-wild,"SAMN10752197,GSM3563901",,,,,,,PMID: 31289822,wild,,SAC,2026-06-10
2,SRX5252126,SRP179635,Illumina HiSeq 2000,SRS4253831,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563900,Liver,Adults,perfect match,not documented,perfect match,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Liver3-RNAseq-wild,"SAMN10752198,GSM3563900",,,,,,,PMID: 31289822,wild,,SAC,2026-06-10
3,SRX5252125,SRP179635,Illumina HiSeq 2000,SRS4253832,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563899,Liver,Adults,perfect match,not documented,perfect match,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Liver2-RNAseq-wild,"SAMN10752199,GSM3563899",,,,,,,PMID: 31289822,wild,,SAC,2026-06-10
4,SRX5252124,SRP179635,Illumina HiSeq 2000,SRS4253830,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563898,Liver,Adults,perfect match,not documented,perfect match,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Liver1-RNAseq-wild,"SAMN10752200,GSM3563898",,,,,,,PMID: 31289822,wild,,SAC,2026-06-10
5,SRX5252123,SRP179635,Illumina HiSeq 2000,SRS4253829,UBERON:0000955,brain,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563897,Brain,Adults,perfect match,not documented,perfect match,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Brain5-RNAseq-wild,"SAMN10752201,GSM3563897",,,,,,,PMID: 31289822,wild,,SAC,2026-06-10
6,SRX5252122,SRP179635,Illumina HiSeq 2000,SRS4253828,UBERON:0000955,brain,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563896,Brain,Adults,perfect match,not documented,perfect match,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Brain4-RNAseq-wild,"SAMN10752202,GSM3563896",,,,,,,PMID: 31289822,wild,,SAC,2026-06-10
7,SRX5252121,SRP179635,Illumina HiSeq 2000,SRS4253827,UBERON:0000955,brain,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563895,Brain,Adults,perfect match,not documented,perfect match,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Brain3-RNAseq-wild,"SAMN10752203,GSM3563895",,,,,,,PMID: 31289822,wild,,SAC,2026-06-10
8,SRX5252120,SRP179635,Illumina HiSeq 2000,SRS4253825,UBERON:0000955,brain,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563894,Brain,Adults,perfect match,not documented,perfect match,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Brain2-RNAseq-wild,"SAMN10752204,GSM3563894",,,,,,,PMID: 31289822,wild,,SAC,2026-06-10
9,SRX5252119,SRP179635,Illumina HiSeq 2000,SRS4253826,UBERON:0000955,brain,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3563893,Brain,Adults,perfect match,not documented,perfect match

### experiment annotations

In [14]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP179635,Epimutations in developmental genes underlie the onset of domestication in farmed European sea bass,"Domestication of wild animals induces a set of phenotypic characteristics collectively known as the domestication syndrome. However, how this syndrome emerges is still not clear. Recently, the neural crest cell deficit hypothesis proposed that it is generated by a mildly disrupted neural crest cell developmental program, but clear support is lacking due to the difficulties of distinguishing pure domestication effects from preexisting genetic differences between farmed and wild mammals and birds. Here, we use a farmed fish as model to investigate the role of persistent changes in DNA methylation (epimutations) in the process of domestication.We show that early domesticates of sea bass, with no genetic differences with wild counterparts, contain epimutations in tissues with different embryonic origins. About one fifth of epimutations that persist into adulthood are established by the time of gastrulation and affect genes involved in developmental processes that are expressed in embryonic structures, including the neural crest. Some of these genes are differentially expressed in sea bass with lower jaw malformations, a key feature of domestication syndrome. Interestingly, these epimutations significantly overlap with cytosine-to-thymine polymorphisms after 25 years of selective breeding. Furthermore, epimutated genes coincide with genes under positive selection in other domesticates. We argue that the initial stages of domestication include dynamic alterations in DNA methylation of developmental genes that affect the neural crest. Our results suggest a role for epimutations during the beginning of domestication that could be fixed as genetic variants and suggest a conserved molecular process to explain Darwin’s domestication syndrome across vertebrates. Overall design: We analyzed genome-wide methylation and gene expression differences between wild European sea bass captured by speargun in the North-Eastern coast of Spain and European sea bass born in a hatchery and reared under farming conditions until adulthood. Genome-wide methylation patterns were analyzed by performing Reduced Representation Sequencing (RRBS) of four tissues representative of the embryonic layers: brain (ectoderm), muscle (paraxial mesoderm), testis (intermediate mesoderm) and liver (endoderm) from three wild and three farmed fish. In addition, we performed RRBS for three pools of five eggs each at 27-30 hours post fertilization spawned and fertilized by captive fish. Gene expression patterns were analyzed by RNA-seq in the brain, muscle, testis and liver of five wild and five farmed fish (seven testes of farmed fish). The RRBS and RNA-seq data of muscle and testis of wild fish are already published under the GEO Series accession number GSE104366.",SRA,,,,TruSeq Stranded mRNA,full_length,GSE125124,PRJNA515290,"31289822, 35741749",,"10.1093/molbev/msz153,10.3390/genes13060987",,


#### experiment and protocol details

In [15]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

32

In [16]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'partial'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
experiment.loc[:,'protocol'] = my_protocol
experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP179635,Epimutations in developmental genes underlie the onset of domestication in farmed European sea bass,"Domestication of wild animals induces a set of phenotypic characteristics collectively known as the domestication syndrome. However, how this syndrome emerges is still not clear. Recently, the neural crest cell deficit hypothesis proposed that it is generated by a mildly disrupted neural crest cell developmental program, but clear support is lacking due to the difficulties of distinguishing pure domestication effects from preexisting genetic differences between farmed and wild mammals and birds. Here, we use a farmed fish as model to investigate the role of persistent changes in DNA methylation (epimutations) in the process of domestication.We show that early domesticates of sea bass, with no genetic differences with wild counterparts, contain epimutations in tissues with different embryonic origins. About one fifth of epimutations that persist into adulthood are established by the time of gastrulation and affect genes involved in developmental processes that are expressed in embryonic structures, including the neural crest. Some of these genes are differentially expressed in sea bass with lower jaw malformations, a key feature of domestication syndrome. Interestingly, these epimutations significantly overlap with cytosine-to-thymine polymorphisms after 25 years of selective breeding. Furthermore, epimutated genes coincide with genes under positive selection in other domesticates. We argue that the initial stages of domestication include dynamic alterations in DNA methylation of developmental genes that affect the neural crest. Our results suggest a role for epimutations during the beginning of domestication that could be fixed as genetic variants and suggest a conserved molecular process to explain Darwin’s domestication syndrome across vertebrates. Overall design: We analyzed genome-wide methylation and gene expression differences between wild European sea bass captured by speargun in the North-Eastern coast of Spain and European sea bass born in a hatchery and reared under farming conditions until adulthood. Genome-wide methylation patterns were analyzed by performing Reduced Representation Sequencing (RRBS) of four tissues representative of the embryonic layers: brain (ectoderm), muscle (paraxial mesoderm), testis (intermediate mesoderm) and liver (endoderm) from three wild and three farmed fish. In addition, we performed RRBS for three pools of five eggs each at 27-30 hours post fertilization spawned and fertilized by captive fish. Gene expression patterns were analyzed by RNA-seq in the brain, muscle, testis and liver of five wild and five farmed fish (seven testes of farmed fish). The RRBS and RNA-seq data of muscle and testis of wild fish are already published under the GEO Series accession number GSE104366.",SRA,partial,Bgee 1K,32,TruSeq Stranded mRNA,full_length,GSE125124,PRJNA515290,"31289822, 35741749",,"10.1093/molbev/msz153,10.3390/genes13060987",,


#### paper and xrefs

In [17]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
experiment.loc[:,'PMID'] = '31289822'
experiment.loc[:,'reference_url'] = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC6759067/'
experiment.loc[:,'DOI'] = '10.1093/molbev/msz153'
experiment.loc[:,'xrefs'] = '35741749[PMID]'

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP179635,Epimutations in developmental genes underlie the onset of domestication in farmed European sea bass,"Domestication of wild animals induces a set of phenotypic characteristics collectively known as the domestication syndrome. However, how this syndrome emerges is still not clear. Recently, the neural crest cell deficit hypothesis proposed that it is generated by a mildly disrupted neural crest cell developmental program, but clear support is lacking due to the difficulties of distinguishing pure domestication effects from preexisting genetic differences between farmed and wild mammals and birds. Here, we use a farmed fish as model to investigate the role of persistent changes in DNA methylation (epimutations) in the process of domestication.We show that early domesticates of sea bass, with no genetic differences with wild counterparts, contain epimutations in tissues with different embryonic origins. About one fifth of epimutations that persist into adulthood are established by the time of gastrulation and affect genes involved in developmental processes that are expressed in embryonic structures, including the neural crest. Some of these genes are differentially expressed in sea bass with lower jaw malformations, a key feature of domestication syndrome. Interestingly, these epimutations significantly overlap with cytosine-to-thymine polymorphisms after 25 years of selective breeding. Furthermore, epimutated genes coincide with genes under positive selection in other domesticates. We argue that the initial stages of domestication include dynamic alterations in DNA methylation of developmental genes that affect the neural crest. Our results suggest a role for epimutations during the beginning of domestication that could be fixed as genetic variants and suggest a conserved molecular process to explain Darwin’s domestication syndrome across vertebrates. Overall design: We analyzed genome-wide methylation and gene expression differences between wild European sea bass captured by speargun in the North-Eastern coast of Spain and European sea bass born in a hatchery and reared under farming conditions until adulthood. Genome-wide methylation patterns were analyzed by performing Reduced Representation Sequencing (RRBS) of four tissues representative of the embryonic layers: brain (ectoderm), muscle (paraxial mesoderm), testis (intermediate mesoderm) and liver (endoderm) from three wild and three farmed fish. In addition, we performed RRBS for three pools of five eggs each at 27-30 hours post fertilization spawned and fertilized by captive fish. Gene expression patterns were analyzed by RNA-seq in the brain, muscle, testis and liver of five wild and five farmed fish (seven testes of farmed fish). The RRBS and RNA-seq data of muscle and testis of wild fish are already published under the GEO Series accession number GSE104366.",SRA,partial,Bgee 1K,32,TruSeq Stranded mRNA,full_length,GSE125124,PRJNA515290,31289822,https://pmc.ncbi.nlm.nih.gov/articles/PMC6759067/,10.1093/molbev/msz153,35741749[PMID],


#### comments

In [18]:
experiment.loc[:,'comment'] = 'rejected Bisulfite Sequencing libraries'

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP179635,Epimutations in developmental genes underlie the onset of domestication in farmed European sea bass,"Domestication of wild animals induces a set of phenotypic characteristics collectively known as the domestication syndrome. However, how this syndrome emerges is still not clear. Recently, the neural crest cell deficit hypothesis proposed that it is generated by a mildly disrupted neural crest cell developmental program, but clear support is lacking due to the difficulties of distinguishing pure domestication effects from preexisting genetic differences between farmed and wild mammals and birds. Here, we use a farmed fish as model to investigate the role of persistent changes in DNA methylation (epimutations) in the process of domestication.We show that early domesticates of sea bass, with no genetic differences with wild counterparts, contain epimutations in tissues with different embryonic origins. About one fifth of epimutations that persist into adulthood are established by the time of gastrulation and affect genes involved in developmental processes that are expressed in embryonic structures, including the neural crest. Some of these genes are differentially expressed in sea bass with lower jaw malformations, a key feature of domestication syndrome. Interestingly, these epimutations significantly overlap with cytosine-to-thymine polymorphisms after 25 years of selective breeding. Furthermore, epimutated genes coincide with genes under positive selection in other domesticates. We argue that the initial stages of domestication include dynamic alterations in DNA methylation of developmental genes that affect the neural crest. Our results suggest a role for epimutations during the beginning of domestication that could be fixed as genetic variants and suggest a conserved molecular process to explain Darwin’s domestication syndrome across vertebrates. Overall design: We analyzed genome-wide methylation and gene expression differences between wild European sea bass captured by speargun in the North-Eastern coast of Spain and European sea bass born in a hatchery and reared under farming conditions until adulthood. Genome-wide methylation patterns were analyzed by performing Reduced Representation Sequencing (RRBS) of four tissues representative of the embryonic layers: brain (ectoderm), muscle (paraxial mesoderm), testis (intermediate mesoderm) and liver (endoderm) from three wild and three farmed fish. In addition, we performed RRBS for three pools of five eggs each at 27-30 hours post fertilization spawned and fertilized by captive fish. Gene expression patterns were analyzed by RNA-seq in the brain, muscle, testis and liver of five wild and five farmed fish (seven testes of farmed fish). The RRBS and RNA-seq data of muscle and testis of wild fish are already published under the GEO Series accession number GSE104366.",SRA,partial,Bgee 1K,32,TruSeq Stranded mRNA,full_length,GSE125124,PRJNA515290,31289822,https://pmc.ncbi.nlm.nih.gov/articles/PMC6759067/,10.1093/molbev/msz153,35741749[PMID],rejected Bisulfite Sequencing libraries


#### save complete file

In [19]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [20]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

In [21]:
! python3 $path_to_v_script --bulk-exp $experiment_to_add_path --bulk-lib $library_to_add_path --rules-dir $path_to_rules --out $val_output --strict

Total issues: 0
Errors: 0
Warnings: 0
Top codes:


#### check columns match

In [24]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [25]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
65345,SRX4213810,SRP150499,NextSeq 500,SRS3416470,UBERON:0000945,stomach,RnorDv:0000058,juvenile stage,,stomach tissues,around 8 to 9 weeks old,perfect match,not documented,other,M,Sprague-Dawley,,10116,TruSeq Stranded mRNA,full_length,polyA,,,K0_S,SAMN09425536,,,,,,,this is actually 4 different parts of the stom...,Non-infected,,SAC,2026-06-08
65346,SRX4213815,SRP150499,NextSeq 500,SRS3416475,UBERON:0002385,muscle tissue,RnorDv:0000058,juvenile stage,,muscle tissues,around 8 to 9 weeks old,perfect match,not documented,other,M,Sprague-Dawley,,10116,TruSeq Stranded mRNA,full_length,polyA,,,K0_M,SAMN09425527,,,,,,,PMID: 30245697,Non-infected,,SAC,2026-06-08
65347,SRX5252128,SRP179635,Illumina HiSeq 2000,SRS4253834,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Liver,Adults,perfect match,not documented,perfect match,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Liver5-RNAseq-wild,"SAMN10752196,GSM3563902",,,,,,,PMID: 31289822,wild,,SAC,2026-06-10
65348,SRX5252127,SRP179635,Illumina HiSeq 2000,SRS4253833,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Liver,Adults,perfect match,not documented,perfect match,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Liver4-RNAseq-wild,"SAMN10752197,GSM3563901",,,,,,,PMID: 31289822,wild,,SAC,2026-06-10
65349,SRX5252126,SRP179635,Illumina HiSeq 2000,SRS4253831,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Liver,Adults,perfect match,not documented,perfect match,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Liver3-RNAseq-wild,"SAMN10752198,GSM3563900",,,,,,,PMID: 31289822,wild,,SAC,2026-06-10
65350,SRX5252125,SRP179635,Illumina HiSeq 2000,SRS4253832,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Liver,Adults,perfect match,not documented,perfect match,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Liver2-RNAseq-wild,"SAMN10752199,GSM3563899",,,,,,,PMID: 31289822,wild,,SAC,2026-06-10
65351,SRX5252124,SRP179635,Illumina HiSeq 2000,SRS4253830,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Liver,Adults,perfect match,not documented,perfect match,NA,,,13489,TruSeq Stranded mRNA,full_length,polyA,,,Liver1-RNAseq-wild,"SAMN10752200,GSM3563898",,,,,,,PMID: 31289822,wild,,SAC,2026-06-10


In [26]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1246,SRP446379,An RNA-seq atlas of mouse brain areas during f...,Mammalian energy homeostasis is regulated by t...,SRA,total,,185,NEBNext Ultra Directional RNA Library Prep Kit,full_length,GSE236077,PRJNA988797,38184639,https://pmc.ncbi.nlm.nih.gov/articles/PMC10771...,10.1038/s41597-023-02888-4,,[Bgee curator notes: mouse fasted max 24 hours...
1247,SRP150499,Rattus norvegicus and Anisakis pegreffii RNAse...,Considered as one of the most significant emer...,SRA,partial,Bgee 1K,2,TruSeq Stranded mRNA,full_length,,PRJNA475982,30245697,https://pmc.ncbi.nlm.nih.gov/articles/PMC6137129/,10.3389/fimmu.2018.02055,"36590595[PMID],34186188[PMID]",rejected infected libraries and cell culture
1248,SRP179635,Epimutations in developmental genes underlie t...,Domestication of wild animals induces a set of...,SRA,partial,Bgee 1K,32,TruSeq Stranded mRNA,full_length,GSE125124,PRJNA515290,31289822,https://pmc.ncbi.nlm.nih.gov/articles/PMC6759067/,10.1093/molbev/msz153,35741749[PMID],rejected Bisulfite Sequencing libraries


### add annotations to git

In [27]:
! git pull

Already up to date.


In [28]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [ ]:
! git status

In [30]:
! git add $git_experiment_path $git_library_path

In [31]:
! git commit -m $commit_message_exp

[develop 854bcca] adding annotated bulk experiment SRP179635
 2 files changed, 33 insertions(+)


In [32]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 4.30 KiB | 1.08 MiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   39b7e85..854bcca  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push